# Step 7: Feature Engineering
Build business-level features on top of cleaned data.

In [ ]:
import pandas as pd
import numpy as np

orders = pd.read_csv('../data/cleaned/orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'
])
order_items = pd.read_csv('../data/cleaned/order_items.csv')
payments = pd.read_csv('../data/cleaned/order_payments.csv')
customers = pd.read_csv('../data/cleaned/customers.csv')

# IMPORTANT: in this dataset customer_id is unique PER ORDER, not per person.
# customer_unique_id is the real customer identity — always use it for
# customer-level aggregation (repeat purchase behavior, lifetime spend, etc.)
orders = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

## Order-level features

In [ ]:
# Total order value = sum(price + freight) per order
order_value = order_items.groupby('order_id').apply(
    lambda x: (x['price'] + x['freight_value']).sum()
).reset_index(name='total_order_value')

orders = orders.merge(order_value, on='order_id', how='left')

# Delivery days = delivered date - purchase date
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

# Delivery delay = delivered date - estimated date (positive = late)
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days
orders['is_delayed'] = orders['delivery_delay_days'] > 0

## Customer-level features

In [ ]:
customer_features = orders.groupby('customer_unique_id').agg(
    customer_order_count=('order_id', 'nunique'),
    customer_total_spending=('total_order_value', 'sum'),
    customer_avg_order_value=('total_order_value', 'mean')
).reset_index()

customer_features['is_repeat_customer'] = customer_features['customer_order_count'] > 1
print(f"Repeat customers: {customer_features['is_repeat_customer'].sum()} / {len(customer_features)}")
customer_features.head()

## Seller-level features

In [ ]:
seller_features = order_items.groupby('seller_id').agg(
    seller_order_count=('order_id', 'nunique'),
    seller_revenue=('price', 'sum')
).reset_index()
seller_features.head()

In [ ]:
# Save engineered features
orders.to_csv('../data/cleaned/orders_features.csv', index=False)
customer_features.to_csv('../data/cleaned/customer_features.csv', index=False)
seller_features.to_csv('../data/cleaned/seller_features.csv', index=False)